<a href="https://colab.research.google.com/github/ipeirotis/dealing_with_data/blob/master/01-Pandas/A4-NYPD_Vehicle_Collisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A4: NYPD Vehicle Collisions — Integration Exercise

## Putting It All Together

In notebooks A1-A3, you learned Pandas skills using NYC Restaurant Inspection data. Now it's time to **apply those skills independently** to a new dataset: NYPD Motor Vehicle Collisions.

This notebook is structured differently from A1-A3:
- You'll be given **tasks** rather than step-by-step instructions
- Solutions are provided, but **try each task yourself first**
- Multiple approaches (SQL and Pandas) are shown—use whichever feels natural

## Learning Objectives

By completing this notebook, you will demonstrate your ability to:

1. Load and filter data from BigQuery using SQL
2. Explore distributions using `value_counts()` and histograms
3. Aggregate data using `pivot_table()` and `groupby()`
4. Join tables to combine information
5. Analyze time series data using `resample()`
6. Create computed columns for feature engineering
7. Visualize patterns using various plot types
8. Draw insights from data to answer real-world questions

## Skills Checklist

Each task maps to skills from previous notebooks:

| Task | Description | Skills from A1-A3 |
|------|-------------|-------------------|
| 1 | Load data with filters | A1: SQL queries, DataFrames |
| 2 | Contributing factors | A1: `value_counts()`, bar charts |
| 3 | Borough breakdown | A1: `value_counts()`, visualization |
| 4 | Injury distribution | A1: histograms, log scales |
| 5 | Average injuries by group | A3: `pivot_table()`, A2: joins |
| 6 | Two-dimensional pivot | A3: 2D pivot tables |
| 7 | Dates with most accidents | A2: `sort_values()` |
| 8 | Time series analysis | A3: `resample()`, datetime |
| 9 | Hour-of-day patterns | A3: `assign()`, feature engineering |
| 10 | Geospatial visualization | A3: `apply()`, advanced viz |
| **Capstone** | Congestion pricing analysis | **All skills combined** |

---

## About the Dataset

The [NYPD Motor Vehicle Collisions](https://data.cityofnewyork.us/Public-Safety/NYPD-Motor-Vehicle-Collisions/h9gi-nx95/data) dataset contains information about every traffic collision reported by NYPD since July 2012.

### Tables in BigQuery

**`nyu-datasets.collisions.collisions`** — Main collision records

| Column | Description |
|--------|-------------|
| `UNIQUE_KEY` | Unique identifier for each collision |
| `DATE_TIME` | When the collision occurred |
| `REPORTED_BOROUGH` | Borough where collision occurred |
| `LATITUDE`, `LONGITUDE` | Geographic coordinates |
| `PERSONS_INJURED` | Total people injured |
| `PERSONS_KILLED` | Total people killed |
| `PEDESTRIANS_INJURED/KILLED` | Pedestrian casualties |
| `CYCLISTS_INJURED/KILLED` | Cyclist casualties |
| `MOTORISTS_INJURED/KILLED` | Motorist casualties |

**`nyu-datasets.collisions.causes_types`** — Contributing factors and vehicle types

| Column | Description |
|--------|-------------|
| `UNIQUE_KEY` | Links to collisions table |
| `CAUSE` | Contributing factor (e.g., "Driver Inattention") |
| `VEHICLE_TYPE` | Type of vehicle involved |

**Note**: One collision can have multiple rows in `causes_types` (multiple vehicles/factors involved).

### Data Volume

- ~2 million collision records (2012-present)
- ~4 million cause/vehicle records
- Updated regularly with new incidents

---

## Setup

In [ ]:
# You need to change the project_id to your own Google Cloud project
project_id = "nyu-datasets"  # <<<<<< CHANGE THIS IF NEEDED

In [ ]:
# @title Setup and preliminaries

# Authentication and BigQuery setup
!pip install -q google-cloud-bigquery

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
client = bigquery.Client(project=project_id)

# Import libraries
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Render plots with high resolution
%config InlineBackend.figure_format = 'retina'

# Set plotting style
matplotlib.style.use(["seaborn-v0_8-talk", "seaborn-v0_8-ticks", "seaborn-v0_8-whitegrid"])
pd.options.plotting.backend = 'matplotlib'
plt.rcParams['figure.figsize'] = [10, 3]

print("✓ Setup complete!")

In [ ]:
# @title Geospatial capabilities setup (for Task 10)

import geopandas as gpd

# NYC neighborhood boundaries from NYC Open Data
df_nyc = gpd.GeoDataFrame.from_file(
    'https://data.cityofnewyork.us/resource/9nt8-h7nd.geojson'
)

print("✓ Geospatial data loaded!")

---

## Quick Warm-Up: Explore the Schema

Before diving into tasks, let's see what we're working with.

In [ ]:
# Preview the collisions table
sql = '''
SELECT *
FROM nyu-datasets.collisions.collisions
LIMIT 5
'''
client.query(sql).to_dataframe()

In [ ]:
# Preview the causes_types table
sql = '''
SELECT *
FROM nyu-datasets.collisions.causes_types
LIMIT 5
'''
client.query(sql).to_dataframe()

In [ ]:
# How much data do we have?
sql = '''
SELECT
    MIN(DATE_TIME) as earliest,
    MAX(DATE_TIME) as latest,
    COUNT(*) as total_collisions
FROM nyu-datasets.collisions.collisions
'''
client.query(sql).to_dataframe()

---

# 📋 Tasks

**Instructions**: For each task, try to write your own solution before looking at the provided solution. You can use either SQL or Pandas—or both!

---

### Task 1: Load the Data

Load collision data for all accidents **after January 1st, 2020**. You'll need to load two tables:

1. `nyu-datasets.collisions.collisions` → save as `c`
2. `nyu-datasets.collisions.causes_types` → save as `v`

**Hint**: For `causes_types`, you'll need a subquery to filter to only the relevant collision IDs.

**Skills**: SQL `WHERE` clause, date filtering, subqueries

In [ ]:
# YOUR CODE HERE: Load collisions table
# sql = '''
#     SELECT *
#     FROM nyu-datasets.collisions.collisions
#     WHERE ...
# '''
# c = client.query(sql).to_dataframe()

In [ ]:
# YOUR CODE HERE: Load causes_types table
# sql = '''
#     SELECT *
#     FROM nyu-datasets.collisions.causes_types
#     WHERE UNIQUE_KEY IN (...)
# '''
# v = client.query(sql).to_dataframe()

#### Solution

In [ ]:
# Load collisions table
sql = '''
SELECT *
FROM nyu-datasets.collisions.collisions
WHERE DATE_TIME >= '2020-01-01'
'''
c = client.query(sql).to_dataframe()
print(f"Loaded {len(c):,} collisions")

In [ ]:
# Load causes_types table (only for collisions we loaded)
sql = '''
SELECT *
FROM nyu-datasets.collisions.causes_types
WHERE UNIQUE_KEY IN (
    SELECT UNIQUE_KEY
    FROM nyu-datasets.collisions.collisions
    WHERE DATE_TIME >= '2020-01-01'
)
'''
v = client.query(sql).to_dataframe()
print(f"Loaded {len(v):,} cause/vehicle records")

---

### Task 2: Contributing Factors

Find the **most common contributing factors** to collisions.

**Skills**: `value_counts()`, bar charts

**Question to consider**: What's the difference between counting rows vs counting unique accidents?

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# Simple approach with value_counts()
v['CAUSE'].value_counts().head(10)

In [ ]:
# Visualization
v['CAUSE'].value_counts().head(10).plot(kind='barh')
plt.xlabel('Number of Occurrences')
plt.title('Most Common Contributing Factors')

In [ ]:
# SQL approach - note the difference between COUNT(*) and COUNT(DISTINCT)
factors_sql = '''
SELECT CAUSE,
       COUNT(*) AS vehicle_count,
       COUNT(DISTINCT UNIQUE_KEY) AS accident_count
FROM nyu-datasets.collisions.causes_types
WHERE UNIQUE_KEY IN (
    SELECT UNIQUE_KEY
    FROM nyu-datasets.collisions.collisions
    WHERE DATE_TIME >= '2020-01-01'
)
GROUP BY CAUSE
ORDER BY accident_count DESC
'''

factors_df = client.query(factors_sql).to_dataframe()
factors_df.head(10)

In [ ]:
# Better visualization
(
    factors_df
    .head(10)
    .sort_values('accident_count')
    .plot(
        kind='barh',
        x='CAUSE',
        y='accident_count',
        figsize=(10, 5),
        legend=False
    )
)
plt.xlabel('Number of Accidents')
plt.ylabel('')
plt.title('Top 10 Contributing Factors to Vehicle Collisions (2020+)')
plt.tight_layout()

---

### Task 3: Collisions by Borough

Break down the number of collisions by borough.

**Skills**: `value_counts()`, bar charts

**Question to consider**: Does the ranking surprise you? What might explain it?

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# Pandas approach
(
    c['REPORTED_BOROUGH']
    .value_counts()
    .plot(
        kind='barh',
        figsize=(8, 3),
        color='steelblue'
    )
)
plt.xlabel('Number of Collisions')
plt.ylabel('')
plt.title('Collisions by Borough (2020+)')

In [ ]:
# SQL approach
boro_sql = '''
SELECT REPORTED_BOROUGH, COUNT(*) AS cnt
FROM nyu-datasets.collisions.collisions
WHERE DATE_TIME >= '2020-01-01'
GROUP BY REPORTED_BOROUGH
ORDER BY cnt DESC
'''

boro_df = client.query(boro_sql).to_dataframe()
boro_df

---

### Task 4: Injury Distribution

Find out how many collisions had 0, 1, 2, etc. persons injured.

**Skills**: `value_counts()`, `sort_index()`, logarithmic scales

**Hint**: The distribution is heavily skewed—try `logy=True` in your plot.

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# Using method chaining
(
    c['PERSONS_INJURED']
    .value_counts()
    .sort_index()  # Sort by number of injuries, not frequency
    .plot(
        kind='line',
        marker='o',
        logy=True,  # Logarithmic y-axis
        figsize=(10, 4)
    )
)
plt.xlabel('Number of Persons Injured')
plt.ylabel('Number of Collisions (log scale)')
plt.title('Distribution of Injuries per Collision')

---

### 🔍 Checkpoint: Understanding Distributions

**Pause and reflect**:
1. Why does the injury distribution follow this pattern?
2. What does the logarithmic scale reveal that a linear scale would hide?
3. Would you expect the same pattern for deaths? Why or why not?

---

### Task 5: Average Injuries and Deaths by Group

**(a)** Compute the average number of injuries and deaths per accident, broken down by **borough**.

**(b)** Compute the same, but broken down by **contributing factor**. Show the 10 deadliest factors.

**Skills**: `pivot_table()`, joins, `sort_values()`

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# Part (a): By borough using Pandas
pd.pivot_table(
    data=c,
    index='REPORTED_BOROUGH',
    values=['PERSONS_INJURED', 'PERSONS_KILLED'],
    aggfunc='mean'
).round(4)

In [ ]:
# Part (a): By borough using SQL
sql = '''
SELECT REPORTED_BOROUGH,
       AVG(PERSONS_INJURED) AS avg_injured,
       AVG(PERSONS_KILLED) AS avg_killed
FROM nyu-datasets.collisions.collisions
WHERE DATE_TIME >= '2020-01-01'
GROUP BY REPORTED_BOROUGH
'''
client.query(sql).to_dataframe().round(4)

In [ ]:
# Part (b): By contributing factor using SQL join
sql = '''
SELECT V.CAUSE,
       AVG(C.PERSONS_INJURED) AS avg_injured,
       AVG(C.PERSONS_KILLED) AS avg_killed,
       COUNT(*) AS num_accidents
FROM nyu-datasets.collisions.collisions C
JOIN nyu-datasets.collisions.causes_types V ON C.UNIQUE_KEY = V.UNIQUE_KEY
WHERE DATE_TIME >= '2020-01-01'
GROUP BY V.CAUSE
'''

cause_severity = client.query(sql).to_dataframe()

# Top 10 deadliest causes
(
    cause_severity
    .sort_values('avg_killed', ascending=False)
    .head(10)
    .round(4)
)

---

### Task 6: Two-Dimensional Pivot Table

Create a pivot table showing the number of accidents by **cause** (rows) and **borough** (columns).

**Skills**: 2D `pivot_table()`, computed columns

**Bonus**: Add a "Total" column and sort by it.

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# First, get the data via SQL
sql = '''
SELECT C.REPORTED_BOROUGH, V.CAUSE, COUNT(DISTINCT C.UNIQUE_KEY) AS cnt
FROM nyu-datasets.collisions.collisions C
JOIN nyu-datasets.collisions.causes_types V ON C.UNIQUE_KEY = V.UNIQUE_KEY
WHERE DATE_TIME >= '2020-01-01'
GROUP BY C.REPORTED_BOROUGH, V.CAUSE
'''

result = client.query(sql).to_dataframe()

In [ ]:
# Create the pivot table
pivot = pd.pivot_table(
    data=result,
    index='CAUSE',
    columns='REPORTED_BOROUGH',
    values='cnt',
    aggfunc='sum'
)

# Add total column and sort
pivot['Total'] = pivot.sum(axis='columns')
pivot = pivot.sort_values('Total', ascending=False)

pivot.head(10)

---

### Task 7: Dates with Most Accidents

Find the dates with the most accidents. Can you figure out what happened on these days?

**Skills**: Grouping by date, `sort_values()`

**Hint**: Search the news for these dates!

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
sql = '''
SELECT DATE(DATE_TIME) AS accident_date, COUNT(*) AS cnt
FROM nyu-datasets.collisions.collisions
GROUP BY DATE(DATE_TIME)
ORDER BY cnt DESC
'''

date_df = client.query(sql).to_dataframe()
date_df.head(20)

---

### Task 8: Time Series Analysis

Plot the number of accidents over time. Use `resample()` to smooth the data to weekly or monthly totals.

**Skills**: `resample()`, datetime conversion, time series visualization

**Hint**: Make sure your date column is a proper datetime type before resampling.

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# Get daily data
sql = '''
SELECT DATE(DATE_TIME) AS accident_date,
       COUNT(*) AS cnt,
       SUM(PERSONS_INJURED) AS persons_injured,
       SUM(PERSONS_KILLED) AS persons_killed
FROM nyu-datasets.collisions.collisions
GROUP BY DATE(DATE_TIME)
'''

date_df = client.query(sql).to_dataframe()

# Convert to datetime
date_df['date'] = pd.to_datetime(date_df['accident_date'])

In [ ]:
# Weekly resampling
(
    pd.pivot_table(
        data=date_df,
        index='date',
        values='cnt'
    )
    .resample('1W').sum()
    .plot(figsize=(15, 4))
)
plt.title('Weekly Collisions Over Time')
plt.ylabel('Number of Collisions')

In [ ]:
# Monthly resampling with injuries
(
    pd.pivot_table(
        data=date_df,
        index='date',
        values='persons_injured'
    )
    .resample('1M').sum()
    .plot(figsize=(15, 4), marker='o', markersize=3)
)
plt.title('Monthly Injuries from Vehicle Collisions')
plt.ylabel('Total Persons Injured')

---

### 🔍 Checkpoint: Time Series Patterns

**Pause and reflect**:
1. What major event is visible in early 2020?
2. Has the number of collisions returned to pre-pandemic levels?
3. What seasonal patterns can you identify?

---

### Task 9: Hour-of-Day Analysis

Analyze whether the **time of day** affects the probability of injury or death.

Steps:
1. Create an `HOUR` column from the datetime
2. Create boolean columns `INJURY` and `DEATH`
3. Calculate the probability of injury/death by hour
4. Visualize the patterns

**Skills**: Feature engineering, boolean columns, `pivot_table()` with mean

**Prediction**: Before looking at the data—what time of day do you think is most dangerous?

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# Get data with computed columns via SQL
sql = '''
SELECT UNIQUE_KEY,
       DATE_TIME,
       EXTRACT(HOUR FROM DATE_TIME) AS HOUR,
       PERSONS_INJURED > 0 AS INJURY,
       PERSONS_KILLED > 0 AS DEATH
FROM nyu-datasets.collisions.collisions
'''

df = client.query(sql).to_dataframe()

In [ ]:
# Probability of injury/death by hour
hourly_risk = pd.pivot_table(
    data=df,
    index='HOUR',
    values=['INJURY', 'DEATH'],
    aggfunc='mean'
)

hourly_risk.plot(
    secondary_y=['DEATH'],  # Different scale for deaths
    figsize=(12, 5),
    marker='o'
)
plt.title('Probability of Injury/Death by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Probability of Injury')

---

### Task 10: Geospatial Visualization

Create a visualization showing the locations of **cyclist fatalities**.

Steps:
1. Filter for accidents where `CYCLISTS_KILLED > 0`
2. Create a scatter plot of longitude/latitude
3. Add a 2D kernel density plot to show hotspots
4. Overlay on NYC map boundaries

**Skills**: Filtering, scatter plots, KDE plots, layered visualization

In [ ]:
# YOUR CODE HERE


#### Solution

In [ ]:
# Get cyclist fatality locations
sql = '''
SELECT LONGITUDE, LATITUDE
FROM nyu-datasets.collisions.collisions
WHERE CYCLISTS_KILLED > 0
'''

cyclist_dead = client.query(sql).to_dataframe()

In [ ]:
# Simple scatter plot
cyclist_dead.plot(
    kind='scatter',
    x='LONGITUDE',
    y='LATITUDE',
    figsize=(10, 10),
    alpha=0.6
)
plt.title('Cyclist Fatality Locations')

In [ ]:
# Full visualization with NYC boundaries and density
fig, ax = plt.subplots(figsize=(12, 12))

# NYC boundaries
df_nyc.plot(
    ax=ax,
    linewidth=0.5,
    color='white',
    edgecolor='black',
    alpha=0.75
)

# Scatter plot
cyclist_dead.plot(
    kind='scatter',
    x='LONGITUDE',
    y='LATITUDE',
    ax=ax,
    alpha=0.6,
    color='red',
    s=20
)

# 2D KDE overlay
sns.kdeplot(
    data=cyclist_dead,
    x='LONGITUDE',
    y='LATITUDE',
    fill=True,
    gridsize=100,
    cmap='Reds',
    alpha=0.5,
    levels=15,
    ax=ax
)

plt.title('Cyclist Fatality Hotspots in NYC', fontsize=14)
plt.xlabel('Longitude')
plt.ylabel('Latitude')

---

# 🎯 Capstone: Congestion Pricing Impact Analysis

## Background

On **January 5th, 2025**, New York City implemented congestion pricing in Manhattan's Central Business District (CBD)—roughly the area south of 60th Street. Vehicles entering this zone are charged a toll, with the goal of reducing traffic and raising funds for public transit.

## Your Challenge

Using the collision dataset, analyze whether congestion pricing has had any measurable impact on vehicle collisions.

### Key Questions

1. **Volume**: Has the number of collisions in the congestion zone decreased since January 5, 2025?

2. **Comparison**: How does the change in Manhattan compare to changes in other boroughs?

3. **Time patterns**: Has the timing of accidents changed (e.g., peak hours)?

4. **Severity**: Has the ratio of injuries/deaths per accident changed?

### Methodological Considerations

- **Before/After comparison**: What time periods should you compare?
- **Control group**: Can other boroughs serve as a control?
- **Seasonality**: How do you account for seasonal patterns?
- **Confounds**: What other factors might explain changes?

### Getting Started

Here's how to filter for the congestion zone (approximate):

In [ ]:
# Congestion pricing zone (approximate boundaries)
# South of 60th Street in Manhattan
# Latitude roughly < 40.764 for 60th St

sql = '''
SELECT *,
       DATE_TIME >= '2025-01-05' AS post_congestion_pricing,
       (REPORTED_BOROUGH = 'Manhattan' AND LATITUDE < 40.764) AS in_congestion_zone
FROM nyu-datasets.collisions.collisions
WHERE DATE_TIME >= '2024-01-01'  -- One year before and after
'''

congestion_df = client.query(sql).to_dataframe()
print(f"Loaded {len(congestion_df):,} collisions for analysis")

In [ ]:
# YOUR ANALYSIS HERE
# Consider:
# - Daily/weekly trends before vs after
# - Comparison: congestion zone vs rest of Manhattan vs other boroughs
# - Time-of-day patterns
# - Statistical significance


---

## Summary

Congratulations! You've completed the A4 integration exercise. You've demonstrated:

- ✅ Loading and filtering data from BigQuery
- ✅ Exploring distributions with `value_counts()` and histograms
- ✅ Aggregating data with `pivot_table()` and `groupby()`
- ✅ Joining tables to combine information
- ✅ Analyzing time series with `resample()`
- ✅ Creating computed columns for feature engineering
- ✅ Visualizing geographic patterns
- ✅ Approaching real-world policy questions with data

### What's Next?

For a deeper dive into the congestion pricing analysis, see the **Congestion Pricing Mini-Project** companion assignment.